# Amazon Product Review Sentiment Analysis

Build a machine-learning classifier that predicts whether a product review expresses **positive** or **negative** sentiment.

This notebook demonstrates text cleaning, TF-IDF feature extraction, train/test splitting, Logistic Regression, evaluation, and predictions on new reviews.

**Dataset note:** The included CSV is a small synthetic demo dataset for learning and testing. For meaningful evaluation, replace it with a properly sourced real review dataset.

## 1. Import libraries

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay, confusion_matrix)

RANDOM_STATE = 42

## 2. Load the review dataset
In Google Colab, you can upload your own CSV. Otherwise, the notebook uses the included demo file. The CSV must contain columns named `review` and `sentiment` (positive/negative).

In [ ]:
from pathlib import Path
DATA_PATH = Path('data/amazon_reviews_demo.csv')
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
    if csv_files:
        DATA_PATH = Path(csv_files[0])
except ImportError:
    pass  # Running outside Colab: use the included demo CSV.

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## 3. Clean and inspect the text
Remove missing and duplicate rows, normalize sentiment labels, and lightly normalize review text. We retain meaningful words rather than removing all stop words.

In [ ]:
required = {'review', 'sentiment'}
if not required.issubset(df.columns):
    raise ValueError(f'CSV must contain these columns: {sorted(required)}')

df = df[['review', 'sentiment']].dropna().drop_duplicates().copy()
df['review'] = df['review'].astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower()
df = df[df['sentiment'].isin(['positive', 'negative'])]
df = df[df['review'].str.len() > 0].copy()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)      # retain English letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)
df = df[df['clean_review'].str.len() > 0].copy()
print('Cleaned dataset shape:', df.shape)
display(df['sentiment'].value_counts().to_frame('count'))
display(df.head())

## 4. Explore sentiment distribution

In [ ]:
df['sentiment'].value_counts().plot(kind='bar', rot=0)
plt.title('Review Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.tight_layout()
plt.show()

## 5. Split into training and testing sets
The test set is held out until evaluation. Stratification keeps the class proportions similar in both sets.

In [ ]:
X = df['clean_review']
y = df['sentiment']
if y.nunique() != 2 or y.value_counts().min() < 2:
    raise ValueError('Need at least two examples of each sentiment class for a stratified split.')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print('Training examples:', len(X_train))
print('Testing examples:', len(X_test))

## 6. Convert text into numerical features with TF-IDF
Fit the vectorizer on training text only to avoid leaking information from the test set.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print('Training feature matrix:', X_train_tfidf.shape)
print('Testing feature matrix:', X_test_tfidf.shape)

## 7. Train a Logistic Regression classifier

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)
print('Model training complete.')

## 8. Evaluate model performance
Accuracy gives the overall fraction of correct predictions. The F1-score balances precision and recall; the classification report also shows per-class metrics.

In [ ]:
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label='positive')
print(f'Accuracy: {accuracy:.3f}')
print(f'F1-score (positive class): {f1:.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=['negative', 'positive'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 9. Predict sentiment for new reviews
Try your own review text. The same cleaning and fitted TF-IDF transformation are applied before prediction.

In [ ]:
def predict_sentiment(review_text):
    cleaned = clean_text(review_text)
    if not cleaned:
        raise ValueError('Please provide a review containing readable text.')
    features = tfidf.transform([cleaned])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    confidence = float(probabilities.max())
    return prediction, confidence

new_reviews = [
    'The product works perfectly and the quality is excellent.',
    'Very disappointing. It stopped working after one day.'
]
for review in new_reviews:
    sentiment, confidence = predict_sentiment(review)
    print(f'Review: {review}')
    print(f'Predicted sentiment: {sentiment} (model confidence: {confidence:.2f})\n')

## 10. Save the trained model components (optional)
Save both the classifier and vectorizer so they can be reused together.

In [ ]:
import joblib
Path('models').mkdir(exist_ok=True)
joblib.dump(model, 'models/sentiment_model.joblib')
joblib.dump(tfidf, 'models/tfidf_vectorizer.joblib')
print('Saved model and vectorizer in the models/ folder.')

## 11. Conclusion
Record the accuracy and F1-score from your real dataset, note any class imbalance or data limitations, and explain how you might improve the model (for example, more labelled data, cross-validation, or comparing other classifiers).

**Reminder:** A high score on a tiny demo dataset does not prove real-world performance. Evaluate on a sufficiently large, representative dataset.